# 08 ML Prediction

Load saved ML models, split the trial feature table 80:20, pick random test samples, predict HR, cardiovascular load, and BP proxy, then save the prediction table.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import FEATURES_DIR, MODELS_DIR, OUTPUT_DIR
from src.models.prediction import predict_ml_samples, random_sample, split_80_20


## 1. Load features and saved ML models

In [ ]:
features = pd.read_csv(FEATURES_DIR / 'trial_features.csv')
model_paths = {
    'hr': MODELS_DIR / 'hr_regressor.joblib',
    'cv_load': MODELS_DIR / 'cv_load_classifier.joblib',
    'bp_proxy': MODELS_DIR / 'bp_proxy_regressor.joblib',
}
features.shape, features.head()


## 2. Split 80:20 and sample from test set

In [ ]:
train_df, test_df = split_80_20(features, random_state=42, stratify_col='cv_load_class')
samples = random_sample(test_df, n=5, random_state=7)
train_df.shape, test_df.shape, samples[['subject_id', 'trial_id', 'hr_mean', 'cv_load_class', 'bp_proxy_score']]


## 3. Predict and save results

In [ ]:
predictions = predict_ml_samples(samples, model_paths)
pred_dir = OUTPUT_DIR / 'predictions'
pred_dir.mkdir(parents=True, exist_ok=True)
prediction_path = pred_dir / 'ml_random_sample_predictions.csv'
predictions.to_csv(prediction_path, index=False)
prediction_path, predictions
